# Detecção de Fraude em Cartão de Crédito — Databricks + PySpark + MLflow

Pipeline em arquitetura medalhão (Bronze → Silver → Gold) para EDA, limpeza e treinamento de um modelo de detecção de fraude sobre o dataset [Credit Card Fraud Detection].

Veja o `README.md` do repositório para contexto completo, arquitetura e como rodar.

In [0]:
%pip install kagglehub

import kagglehub
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print(path)

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


100%|██████████| 66.0M/66.0M [00:00<00:00, 96.1MB/s]

Extracting files...


/home/spark-54b91258-68a0-4887-b2e9-2e/.cache/kagglehub/datasets/mlg-ulb/creditcardfraud/versions/3


In [0]:
import pandas as pd

pdf = pd.read_csv(f"{path}/creditcard.csv")
df = spark.createDataFrame(pdf)
df.write.format("delta").mode("overwrite").saveAsTable("bronze_fraude")
display(df.limit(5))

Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0.0,-1.3598071336738,-0.0727811733098497,2.53634673796914,1.37815522427443,-0.338320769942518,0.462387777762292,0.239598554061257,0.0986979012610507,0.363786969611213,0.0907941719789316,-0.551599533260813,-0.617800855762348,-0.991389847235408,-0.311169353699879,1.46817697209427,-0.470400525259478,0.207971241929242,0.0257905801985591,0.403992960255733,0.251412098239705,-0.018306777944153,0.277837575558899,-0.110473910188767,0.0669280749146731,0.128539358273528,-0.189114843888824,0.133558376740387,-0.0210530534538215,149.62,0
0.0,1.19185711131486,0.26615071205963,0.16648011335321,0.448154078460911,0.0600176492822243,-0.0823608088155687,-0.0788029833323113,0.0851016549148104,-0.255425128109186,-0.166974414004614,1.61272666105479,1.06523531137287,0.48909501589608,-0.143772296441519,0.635558093258208,0.463917041022171,-0.114804663102346,-0.183361270123994,-0.145783041325259,-0.0690831352230203,-0.225775248033138,-0.638671952771851,0.101288021253234,-0.339846475529127,0.167170404418143,0.125894532368176,-0.0089830991432281,0.0147241691924927,2.69,0
1.0,-1.35835406159823,-1.34016307473609,1.77320934263119,0.379779593034328,-0.503198133318193,1.80049938079263,0.791460956450422,0.247675786588991,-1.51465432260583,0.207642865216696,0.624501459424895,0.066083685268831,0.717292731410831,-0.165945922763554,2.34586494901581,-2.89008319444231,1.10996937869599,-0.121359313195888,-2.26185709530414,0.524979725224404,0.247998153469754,0.771679401917229,0.909412262347719,-0.689280956490685,-0.327641833735251,-0.139096571514147,-0.0553527940384261,-0.0597518405929204,378.66,0
1.0,-0.966271711572087,-0.185226008082898,1.79299333957872,-0.863291275036453,-0.0103088796030823,1.24720316752486,0.23760893977178,0.377435874652262,-1.38702406270197,-0.0549519224713749,-0.226487263835401,0.178228225877303,0.507756869957169,-0.28792374549456,-0.631418117709045,-1.0596472454325,-0.684092786345479,1.96577500349538,-1.2326219700892,-0.208037781160366,-0.108300452035545,0.0052735967825345,-0.190320518742841,-1.17557533186321,0.647376034602038,-0.221928844458407,0.0627228487293033,0.0614576285006353,123.5,0
2.0,-1.15823309349523,0.877736754848451,1.548717846511,0.403033933955121,-0.407193377311653,0.0959214624684256,0.592940745385545,-0.270532677192282,0.817739308235294,0.753074431976354,-0.822842877946363,0.53819555014995,1.3458515932154,-1.11966983471731,0.175121130008994,-0.451449182813529,-0.237033239362776,-0.0381947870352842,0.803486924960175,0.408542360392758,-0.0094306971323291,0.79827849458971,-0.137458079619063,0.141266983824769,-0.206009587619756,0.502292224181569,0.219422229513348,0.215153147499206,69.99,0


In [0]:
%sql
Select * from bronze_fraude where Class=1 limit 1


Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
406.0,-2.3122265423263,1.95199201064158,-1.60985073229769,3.9979055875468,-0.522187864667764,-1.42654531920595,-2.53738730624579,1.39165724829804,-2.77008927719433,-2.77227214465915,3.20203320709635,-2.89990738849473,-0.595221881324605,-4.28925378244217,0.389724120274487,-1.14074717980657,-2.83005567450437,-0.0168224681808257,0.416955705037907,0.126910559061474,0.517232370861764,-0.0350493686052974,-0.465211076182388,0.320198198514526,0.0445191674731724,0.177839798284401,0.261145002567677,-0.143275874698919,0.0,1


In [0]:
%sql
describe bronze_fraude

col_name,data_type,comment
Time,double,null
V1,double,null
V2,double,null
V3,double,null
V4,double,null
V5,double,null
V6,double,null
V7,double,null
V8,double,null
V9,double,null


In [0]:
%sql
select count(Class) from bronze_fraude   group By Class

count(Class)
284315
492


A EDA serve pra responder as  perguntas

In [0]:
%sql
SELECT Class, COUNT(*) AS total, 
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 3) AS pct
FROM bronze_fraude
GROUP BY Class

Class,total,pct
0,284315,99.827
1,492,0.173


In [0]:
%sql
SELECT 
  SUM(CASE WHEN Amount IS NULL THEN 1 ELSE 0 END) AS nulos_amount,
  SUM(CASE WHEN Time IS NULL THEN 1 ELSE 0 END) AS nulos_time
FROM bronze_fraude

nulos_amount,nulos_time
0,0


In [0]:
%sql
SELECT Class, 
  ROUND(AVG(Amount), 2) AS media, 
  ROUND(MIN(Amount), 2) AS minimo, 
  ROUND(MAX(Amount), 2) AS maximo,
  COUNT(*) AS total
FROM bronze_fraude
GROUP BY Class

Class,media,minimo,maximo,total
0,88.29,0.0,25691.16,284315
1,122.21,0.0,2125.87,492


In [0]:
%sql
SELECT ROUND(Time/3600, 0) AS hora, Class, COUNT(*) AS total
FROM bronze_fraude
GROUP BY hora, Class
ORDER BY hora

hora,Class,total
0.0,0,2258
0.0,1,2
1.0,1,1
1.0,0,3161
2.0,1,21
2.0,0,1491
3.0,0,2093
3.0,1,10
4.0,0,1245
4.0,1,5


In [0]:
%sql
SELECT MOD(CAST(Time/3600 AS INT), 24) AS hora_do_dia, Class, COUNT(*) AS total
FROM bronze_fraude
GROUP BY hora_do_dia, Class
ORDER BY hora_do_dia

hora_do_dia,Class,total
0,0,7689
0,1,6
1,0,4210
1,1,10
2,0,3271
2,1,57
3,0,3475
3,1,17
4,1,23
4,0,2186


In [0]:
%sql
SELECT 
  MOD(CAST(Time/3600 AS INT), 24) AS hora_do_dia,
  SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END) AS normais,
  SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END) AS fraudes
FROM bronze_fraude
GROUP BY hora_do_dia
ORDER BY hora_do_dia

hora_do_dia,normais,fraudes
0,7689,6
1,4210,10
2,3271,57
3,3475,17
4,2186,23
5,2979,11
6,4092,9
7,7220,23
8,10267,9
9,15822,16


In [0]:
import pandas as pd
from sklearn.preprocessing import StandardScaler as SkScaler

df = spark.table("bronze_fraude")
pdf = df.toPandas()

feature_cols = [c for c in pdf.columns if c != "Class"]

scaler = SkScaler()
pdf[feature_cols] = scaler.fit_transform(pdf[feature_cols])

display(pdf.head(5))

Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
-1.996583023457193,-0.6942423209592997,-0.04407492457044802,1.6727734992241514,0.973365514375461,-0.24511658354252203,0.3470679451619034,0.19367893831762997,0.08263727941086944,0.3311277832099309,0.08338554524255039,-0.540407035731003,-0.6182957177945313,-0.9960989219768982,-0.32461018632700356,1.6040138389062168,-0.5368328685192513,0.2448634540209013,0.030769932602201004,0.49628202665105675,0.3261180160164485,-0.024923364961491872,0.38285443833968436,-0.17691133433749112,0.1105069205607409,0.2465854429694212,-0.3921704315485547,0.33089162264871697,-0.06378115069750527,0.2449642633701733,0
-1.996583023457193,0.6084963276928689,0.16117591988327584,0.1097971021060264,0.31652292674812704,0.04348335204726374,-0.06181996595114918,-0.06370020974401087,0.0712534830596023,-0.232494188940568,-0.1533496286145045,1.5800028495181633,1.0660885709726662,0.4914182038851102,-0.14998248182742244,0.6943604186631746,0.5294337538197205,-0.1351699690984707,-0.2187625823123097,-0.1790860493075344,-0.08961086263099204,-0.3073768045456866,-0.8800767544487169,0.1622011836251244,-0.5611305498033157,0.3206939009070386,0.26106947542126635,-0.022255678187035617,0.044607517680124775,-0.3424745411051305,0
-1.9965619655334634,-0.6935004627168447,-0.811577826309909,1.1694684928233277,0.2682312938426664,-0.3645717858377013,1.351453585951466,0.6397756379029848,0.20737272949225674,-1.3786753514283785,0.19069961380161685,0.6118297100478468,0.0661366186801109,0.7206998523261392,-0.173113888845362,2.5629061849805295,-3.298235372241395,1.3068678794417399,-0.14478999149835622,-2.778560850520872,0.6809749715218957,0.33763169617058825,1.0633582711230263,1.456319745720107,-1.1380921384645095,-0.6285367205194723,-0.28844675201538367,-0.1371368556920797,-0.18102082710565728,1.1606859252297228,0
-1.9965619655334634,-0.4933248981469482,-0.11216942463929837,1.1825164508596404,-0.6097266412236745,-0.007468880343610853,0.9361498321757146,0.19207063819763437,0.316017599511344,-1.262503172205911,-0.05046795314643688,-0.22189161429591525,0.17837098770941834,0.5101687012441243,-0.30036049360793543,-0.6898374093803747,-1.2092959931757419,-0.8054446421603219,2.34530452215363,-1.514204923331477,-0.26985522543866364,-0.1474432966746349,0.007266907401368869,-0.30477654737727394,-1.9410271396122685,1.2419037126390189,-0.46021734156146177,0.15539620725948264,0.1861885865396321,0.14053425198451389,0
-1.9965409076097336,-0.5913297637052182,0.5315410497431425,1.0214116755556233,0.2846554042136494,-0.2950154361044151,0.07199858317395955,0.47930228336092434,-0.22651023121331157,0.7443262870821313,0.6916250322008003,-0.8061465859382945,0.5386266478753359,1.352244351584952,-1.1680335140679103,0.19132347214153025,-0.5152051215648076,-0.27908078621696836,-0.045569002860077173,0.9870372970223138,0.5299387935696487,-0.012839217648641762,1.1000112712256316,-0.22012339600658185,0.23325008793816754,-0.39520164174981287,1.0416112996215627,0.5436197963831321,0.6518159160992681,-0.07340334025310612,0


In [0]:
%sql
  SELECT
    MIN(Time) AS t_min,
    MAX(Time) AS t_max,
    ROUND(MAX(Time)/3600, 1) AS horas,
    ROUND(MAX(Time)/86400, 2) AS dias
  FROM bronze_fraude


t_min,t_max,horas,dias
0.0,172792.0,48.0,2.0


In [0]:
%sql
  SELECT COUNT(*) AS total,
         COUNT(DISTINCT Time, Amount, V1, V2, V3) AS distintas
  FROM bronze_fraude



total,distintas
284807,283726


In [0]:
%sql
  SELECT COUNT(*) AS total,
         COUNT(DISTINCT
           Time, Amount, Class,
           V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,
           V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28
         ) AS distintas_full
  FROM bronze_fraude


total,distintas_full
284807,283726


In [0]:
%sql
  -- DESCRIBE mostra o nome e o TIPO de cada coluna.
  -- Precisamos disso porque, se o CSV foi lido sem inferência de schema,
  -- os números podem ter virado STRING — o que quebraria as contas depois.
  DESCRIBE TABLE workspace.default.bronze_fraude;



col_name,data_type,comment
Time,double,null
V1,double,null
V2,double,null
V3,double,null
V4,double,null
V5,double,null
V6,double,null
V7,double,null
V8,double,null
V9,double,null


In [0]:
%sql
  SELECT
    COUNT(*)                                    AS total_linhas,
    -- Cada linha abaixo conta quantos NULLs existem naquela coluna.
    -- SUM(CASE WHEN ... THEN 1 ELSE 0 END) é a forma clássica de contar condicionalmente.
    SUM(CASE WHEN Time   IS NULL THEN 1 ELSE 0 END) AS nulos_time,
    SUM(CASE WHEN Amount IS NULL THEN 1 ELSE 0 END) AS nulos_amount,
    SUM(CASE WHEN Class  IS NULL THEN 1 ELSE 0 END) AS nulos_class,
    SUM(CASE WHEN V1     IS NULL THEN 1 ELSE 0 END) AS nulos_v1,
    SUM(CASE WHEN V28    IS NULL THEN 1 ELSE 0 END) AS nulos_v28
  FROM workspace.default.bronze_fraude;



total_linhas,nulos_time,nulos_amount,nulos_class,nulos_v1,nulos_v28
284807,0,0,0,0,0


In [0]:
colunas = ["Time", "Amount", "Class"] + [f"V{i}" for i in range(1, 29)]
partes = [f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS nulos_{c.lower()}" for c in colunas]

query = f"""
SELECT
  COUNT(*) AS total_linhas,
  {','.join(partes)}
FROM workspace.default.bronze_fraude
"""

display(spark.sql(query))

total_linhas,nulos_time,nulos_amount,nulos_class,nulos_v1,nulos_v2,nulos_v3,nulos_v4,nulos_v5,nulos_v6,nulos_v7,nulos_v8,nulos_v9,nulos_v10,nulos_v11,nulos_v12,nulos_v13,nulos_v14,nulos_v15,nulos_v16,nulos_v17,nulos_v18,nulos_v19,nulos_v20,nulos_v21,nulos_v22,nulos_v23,nulos_v24,nulos_v25,nulos_v26,nulos_v27,nulos_v28
284807,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
df = spark.table("workspace.default.bronze_fraude")

df.select([
    F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
    for c in df.columns
]).show()

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [0]:
%sql
  SELECT
    Class,                                        -- 0 = transação normal, 1 = fraude
    COUNT(*) AS qtd,
    -- ROUND(..., 4) para o percentual não sair como 0.0 (fraude é raríssima)
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS percentual
  FROM workspace.default.bronze_fraude
  GROUP BY Class
  ORDER BY Class;
  


Class,qtd,percentual
0,284315,99.8273
1,492,0.1727


In [0]:
%sql
  SELECT
    MIN(Amount) AS valor_min,
    MAX(Amount) AS valor_max,
    ROUND(AVG(Amount), 2) AS valor_medio,
    -- percentile_approx dá a mediana (0.5) de forma eficiente em dados grandes.
    ROUND(percentile_approx(Amount, 0.5), 2) AS valor_mediana,
    MIN(Time) AS tempo_min,
    MAX(Time) AS tempo_max   -- Time = segundos desde a 1 transação do dataset
  FROM workspace.default.bronze_fraude;



valor_min,valor_max,valor_medio,valor_mediana,tempo_min,tempo_max
0.0,25691.16,88.35,22.0,0.0,172792.0


In [0]:
%sql
  -- Diagnóstico consolidado NULLs, duplicatas reais e desbalanceamento.
  -- UNION ALL empilha três consultas independentes num único resultado.
  -- Todas devolvem 3 colunas (metrica, valor_a, valor_b) para poderem
  -- ser empilhadas — UNION exige mesmo número e tipo de colunas.

  SELECT 'nulos_em_time_amount_class' AS metrica,
         SUM(CASE WHEN Time IS NULL OR Amount IS NULL OR Class IS NULL
                  THEN 1 ELSE 0 END)  AS valor_a,
         COUNT(*)                     AS valor_b
  FROM workspace.default.bronze_fraude

  UNION ALL

  -- DISTINCT * trata NULL como valor comparável, então este é o número
  -- real de duplicatas — diferente de COUNT(DISTINCT col1, col2, ...).
  SELECT 'linhas_distintas_reais',
         (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM workspace.default.bronze_fraude)),
         COUNT(*)
  FROM workspace.default.bronze_fraude

  UNION ALL

  SELECT 'fraudes_class_1',
         SUM(CASE WHEN Class = 1 THEN 1 ELSE 0 END),
         SUM(CASE WHEN Class = 0 THEN 1 ELSE 0 END)
  FROM workspace.default.bronze_fraude

  UNION ALL

  -- Quantas transações de valor exatamente zero, e quantas dessas são fraude
  SELECT 'amount_zero_total_e_fraudes',
         SUM(CASE WHEN Amount = 0 THEN 1 ELSE 0 END),
         SUM(CASE WHEN Amount = 0 AND Class = 1 THEN 1 ELSE 0 END)
  FROM workspace.default.bronze_fraude;


metrica,valor_a,valor_b
nulos_em_time_amount_class,0,284807
fraudes_class_1,492,284315
amount_zero_total_e_fraudes,1825,27
linhas_distintas_reais,283726,284807


In [0]:
%sql
  -- Agrupa por TODAS as colunas relevantes e conta quantas vezes cada
  -- combinação aparece. HAVING COUNT(*) > 1 mantém só as repetidas.
  -- Depois somamos por classe, para saber se a duplicação atinge
  -- fraudes e legítimas na mesma proporção.
  WITH repetidas AS (
    SELECT Time, Amount, Class, COUNT(*) AS vezes
    FROM workspace.default.bronze_fraude
    GROUP BY Time, Amount, Class
    HAVING COUNT(*) > 1
  )
  SELECT
    Class,
    COUNT(*)              AS grupos_duplicados,
    SUM(vezes)            AS linhas_envolvidas,
    SUM(vezes) - COUNT(*) AS linhas_a_remover
  FROM repetidas
  GROUP BY Class
  ORDER BY Class;



Class,grupos_duplicados,linhas_envolvidas,linhas_a_remover
0,3866,8704,4838
1,13,32,19


In [0]:
%sql
SELECT * FROM workspace.default.bronze_fraude
WHERE Time = 19914 AND Amount = 5 AND Class = 0

Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
19914.0,-1.03448232595765,0.75994473694329,1.34467333214426,-1.55424449313282,-0.532825515302296,0.220303026209161,-1.1655522307925,-2.48709461592401,0.0548889853728043,-0.668947069379951,1.25724392043396,-2.84372861554008,0.281256306188759,1.57701133182995,-2.52618507333622,1.19765266994599,0.983546058096941,-0.640644599643313,0.41662034442768,0.689235733396798,-1.37178854451404,0.378573058124428,-0.0397428935881618,-0.0807766069833094,0.0706217761108173,-0.440477283618114,0.263666044896942,0.179713530514989,5.0,0
19914.0,1.40569296761931,-0.45505761122312,0.640889499952696,-0.248345539971386,-1.06801849497726,-0.784040104907807,-0.682285157753401,-0.220919295343124,0.91299803363428,0.136481991960182,0.275076275895739,-3.08935555714058,1.21398251360653,1.05926007666205,-1.17581338725582,0.56842548632104,1.43895793688241,-1.43970475736393,0.740562563593792,-0.0373660151840986,-0.180267477527653,-0.190712759947457,-0.0108780888913218,0.351078610374535,0.52043600996309,-0.281235288578146,-0.0150176585891398,0.0050656411612116,5.0,0
19914.0,1.42411717796772,-0.495474997994606,0.308426512499935,-0.572271349033493,-0.546665211090953,0.0475671039489355,-0.772449390676845,-0.0445350182606686,0.642803283224896,0.293055519790533,1.27775401363277,-2.44113221832095,1.74244923959007,1.11205980883751,-1.86356243531288,0.995799141393401,0.846966499021451,-0.718355220387906,1.35341543949661,0.0313467832666263,-0.180209004650092,-0.199961441135866,-0.135928258264616,-0.557044785711034,0.621530773785195,-0.24653179455173,-0.0169545074053346,-0.0129047743224621,5.0,0
19914.0,1.27601095064793,-0.547342131264456,0.62393594525837,-0.470525045960677,-0.568852834920904,0.501482071827603,-0.889601840299811,0.161101329661102,0.529065727003975,0.193980210542405,2.55618983581269,-1.8185312891651,1.85618778466974,1.16045801483729,-1.45402305619153,0.13821088569798,1.70709390403971,-1.94430949984752,0.301017752312921,-0.041218572529309,-0.0636808146143241,0.216159253930332,0.0277339125115204,-0.300367929498332,0.368807159237695,-0.241935998662329,0.0193671411111679,-0.0102572983572635,5.0,0
19914.0,1.3832213319992,-0.374322572705347,0.8013874878582,-0.179139278300606,-1.14362383379193,-0.955257153072016,-0.606964814357481,-0.29996924333716,0.740698066505206,0.107019407517051,0.736722662801087,-2.45338201895804,2.14797746716525,0.881791989037497,-1.3093229465559,0.42611786519335,1.50151663195615,-1.66862843688839,0.610509107434491,0.0150100714229332,-0.147277553262847,-0.0482778196122065,0.0169406300812118,0.691634394346714,0.510036518980358,-0.304183368130678,-0.0080145821287395,0.0103777535617432,5.0,0
19914.0,1.4043288801027,-0.589821574354749,0.602211985568437,-0.230269357857081,-1.23403317374756,-0.82829818125613,-0.808491432755844,-0.115237832167687,1.19466196656086,0.191118022985743,-0.0421574993573741,-4.07868735763338,-0.646228547886017,1.42946514555525,-0.971716342700956,0.655684208087718,1.54525856663996,-1.21147921257323,0.753121944273782,-0.156550696880254,-0.201655393179536,-0.343059795413449,0.0114499383763353,0.30014764351831,0.457494760713557,-0.275001589356924,-0.0271783120673688,0.0010705063139755,5.0,0
19914.0,-0.515425742222772,-0.377471797494264,1.90381634528886,-0.0719693139416316,-0.314139757363762,-1.11685454251755,-0.583850048889903,-0.092523558016707,0.285986382992853,0.0263967244759626,0.398356288771526,-3.5045123663706,0.799950592328616,1.16251737272034,-0.834158964197737,-0.0147692798163684,1.95854975262699,-1.17903988696488,1.74932000589951,0.272848068134969,0.003110215406888,0.148909378887346,0.0430113094247351,0.672869881084047,-0.328234524169491,-0.228852257060805,0.108071666721303,0.147007630697039,5.0,0
19914.0,1.34816725996528,-0.531812642089039,0.737667065413582,-0.312409779034368,-1.14776721368531,-0.585711320970058,-0.810411843313548,-0.0621158594050208,0.70520830292531,0.317810381322112,2.08121823

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bronze_fraude_com_id AS
SELECT
  monotonically_increasing_id() AS row_id,
  *
FROM workspace.default.bronze_fraude

In [0]:
%sql
SELECT * FROM bronze_fraude_com_id
WHERE Time = 19914 AND Amount = 5 AND Class = 0

row_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
11646,19914.0,-1.03448232595765,0.75994473694329,1.34467333214426,-1.55424449313282,-0.532825515302296,0.220303026209161,-1.1655522307925,-2.48709461592401,0.0548889853728043,-0.668947069379951,1.25724392043396,-2.84372861554008,0.281256306188759,1.57701133182995,-2.52618507333622,1.19765266994599,0.983546058096941,-0.640644599643313,0.41662034442768,0.689235733396798,-1.37178854451404,0.378573058124428,-0.0397428935881618,-0.0807766069833094,0.0706217761108173,-0.440477283618114,0.263666044896942,0.179713530514989,5.0,0
11648,19914.0,1.40569296761931,-0.45505761122312,0.640889499952696,-0.248345539971386,-1.06801849497726,-0.784040104907807,-0.682285157753401,-0.220919295343124,0.91299803363428,0.136481991960182,0.275076275895739,-3.08935555714058,1.21398251360653,1.05926007666205,-1.17581338725582,0.56842548632104,1.43895793688241,-1.43970475736393,0.740562563593792,-0.0373660151840986,-0.180267477527653,-0.190712759947457,-0.0108780888913218,0.351078610374535,0.52043600996309,-0.281235288578146,-0.0150176585891398,0.0050656411612116,5.0,0
11649,19914.0,1.42411717796772,-0.495474997994606,0.308426512499935,-0.572271349033493,-0.546665211090953,0.0475671039489355,-0.772449390676845,-0.0445350182606686,0.642803283224896,0.293055519790533,1.27775401363277,-2.44113221832095,1.74244923959007,1.11205980883751,-1.86356243531288,0.995799141393401,0.846966499021451,-0.718355220387906,1.35341543949661,0.0313467832666263,-0.180209004650092,-0.199961441135866,-0.135928258264616,-0.557044785711034,0.621530773785195,-0.24653179455173,-0.0169545074053346,-0.0129047743224621,5.0,0
11652,19914.0,1.27601095064793,-0.547342131264456,0.62393594525837,-0.470525045960677,-0.568852834920904,0.501482071827603,-0.889601840299811,0.161101329661102,0.529065727003975,0.193980210542405,2.55618983581269,-1.8185312891651,1.85618778466974,1.16045801483729,-1.45402305619153,0.13821088569798,1.70709390403971,-1.94430949984752,0.301017752312921,-0.041218572529309,-0.0636808146143241,0.216159253930332,0.0277339125115204,-0.300367929498332,0.368807159237695,-0.241935998662329,0.0193671411111679,-0.0102572983572635,5.0,0
11653,19914.0,1.3832213319992,-0.374322572705347,0.8013874878582,-0.179139278300606,-1.14362383379193,-0.955257153072016,-0.606964814357481,-0.29996924333716,0.740698066505206,0.107019407517051,0.736722662801087,-2.45338201895804,2.14797746716525,0.881791989037497,-1.3093229465559,0.42611786519335,1.50151663195615,-1.66862843688839,0.610509107434491,0.0150100714229332,-0.147277553262847,-0.0482778196122065,0.0169406300812118,0.691634394346714,0.510036518980358,-0.304183368130678,-0.0080145821287395,0.0103777535617432,5.0,0
11654,19914.0,1.4043288801027,-0.589821574354749,0.602211985568437,-0.230269357857081,-1.23403317374756,-0.82829818125613,-0.808491432755844,-0.115237832167687,1.19466196656086,0.191118022985743,-0.0421574993573741,-4.07868735763338,-0.646228547886017,1.42946514555525,-0.971716342700956,0.655684208087718,1.54525856663996,-1.21147921257323,0.753121944273782,-0.156550696880254,-0.201655393179536,-0.343059795413449,0.0114499383763353,0.30014764351831,0.457494760713557,-0.275001589356924,-0.0271783120673688,0.0010705063139755,5.0,0
11658,19914.0,-0.515425742222772,-0.377471797494264,1.90381634528886,-0.0719693139416316,-0.314139757363762,-1.11685454251755,-0.583850048889903,-0.092523558016707,0.285986382992853,0.0263967244759626,0.398356288771526,-3.5045123663706,0.799950592328616,1.16251737272034,-0.834158964197737,-0.0147692798163684,1.95854975262699,-1.17903988696488,1.74932000589951,0.272848068134969,0.003110215406888,0.148909378887346,0.0430113094247351,0.672869881084047,-0.328234524169491,-0.228852257060805,0.108071666721303,0.147007630697039,5.0,0
11659,19914.0,1.34816725996528,-0.531812642089039,0.737667065413582,-0.312409779034368,-1.14776721368531,-0.585711320970058,-0.810411843313548,-0.0621158

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bronze_fraude_com_id AS
SELECT
  ROW_NUMBER() OVER (ORDER BY Time) AS row_id,
  *
FROM workspace.default.bronze_fraude

In [0]:
%sql
SELECT * FROM bronze_fraude_com_id
WHERE Time = 19914 AND Amount = 5 AND Class = 0

row_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
11647,19914.0,-1.03448232595765,0.75994473694329,1.34467333214426,-1.55424449313282,-0.532825515302296,0.220303026209161,-1.1655522307925,-2.48709461592401,0.0548889853728043,-0.668947069379951,1.25724392043396,-2.84372861554008,0.281256306188759,1.57701133182995,-2.52618507333622,1.19765266994599,0.983546058096941,-0.640644599643313,0.41662034442768,0.689235733396798,-1.37178854451404,0.378573058124428,-0.0397428935881618,-0.0807766069833094,0.0706217761108173,-0.440477283618114,0.263666044896942,0.179713530514989,5.0,0
11649,19914.0,1.40569296761931,-0.45505761122312,0.640889499952696,-0.248345539971386,-1.06801849497726,-0.784040104907807,-0.682285157753401,-0.220919295343124,0.91299803363428,0.136481991960182,0.275076275895739,-3.08935555714058,1.21398251360653,1.05926007666205,-1.17581338725582,0.56842548632104,1.43895793688241,-1.43970475736393,0.740562563593792,-0.0373660151840986,-0.180267477527653,-0.190712759947457,-0.0108780888913218,0.351078610374535,0.52043600996309,-0.281235288578146,-0.0150176585891398,0.0050656411612116,5.0,0
11650,19914.0,1.42411717796772,-0.495474997994606,0.308426512499935,-0.572271349033493,-0.546665211090953,0.0475671039489355,-0.772449390676845,-0.0445350182606686,0.642803283224896,0.293055519790533,1.27775401363277,-2.44113221832095,1.74244923959007,1.11205980883751,-1.86356243531288,0.995799141393401,0.846966499021451,-0.718355220387906,1.35341543949661,0.0313467832666263,-0.180209004650092,-0.199961441135866,-0.135928258264616,-0.557044785711034,0.621530773785195,-0.24653179455173,-0.0169545074053346,-0.0129047743224621,5.0,0
11653,19914.0,1.27601095064793,-0.547342131264456,0.62393594525837,-0.470525045960677,-0.568852834920904,0.501482071827603,-0.889601840299811,0.161101329661102,0.529065727003975,0.193980210542405,2.55618983581269,-1.8185312891651,1.85618778466974,1.16045801483729,-1.45402305619153,0.13821088569798,1.70709390403971,-1.94430949984752,0.301017752312921,-0.041218572529309,-0.0636808146143241,0.216159253930332,0.0277339125115204,-0.300367929498332,0.368807159237695,-0.241935998662329,0.0193671411111679,-0.0102572983572635,5.0,0
11654,19914.0,1.3832213319992,-0.374322572705347,0.8013874878582,-0.179139278300606,-1.14362383379193,-0.955257153072016,-0.606964814357481,-0.29996924333716,0.740698066505206,0.107019407517051,0.736722662801087,-2.45338201895804,2.14797746716525,0.881791989037497,-1.3093229465559,0.42611786519335,1.50151663195615,-1.66862843688839,0.610509107434491,0.0150100714229332,-0.147277553262847,-0.0482778196122065,0.0169406300812118,0.691634394346714,0.510036518980358,-0.304183368130678,-0.0080145821287395,0.0103777535617432,5.0,0
11655,19914.0,1.4043288801027,-0.589821574354749,0.602211985568437,-0.230269357857081,-1.23403317374756,-0.82829818125613,-0.808491432755844,-0.115237832167687,1.19466196656086,0.191118022985743,-0.0421574993573741,-4.07868735763338,-0.646228547886017,1.42946514555525,-0.971716342700956,0.655684208087718,1.54525856663996,-1.21147921257323,0.753121944273782,-0.156550696880254,-0.201655393179536,-0.343059795413449,0.0114499383763353,0.30014764351831,0.457494760713557,-0.275001589356924,-0.0271783120673688,0.0010705063139755,5.0,0
11659,19914.0,-0.515425742222772,-0.377471797494264,1.90381634528886,-0.0719693139416316,-0.314139757363762,-1.11685454251755,-0.583850048889903,-0.092523558016707,0.285986382992853,0.0263967244759626,0.398356288771526,-3.5045123663706,0.799950592328616,1.16251737272034,-0.834158964197737,-0.0147692798163684,1.95854975262699,-1.17903988696488,1.74932000589951,0.272848068134969,0.003110215406888,0.148909378887346,0.0430113094247351,0.672869881084047,-0.328234524169491,-0.228852257060805,0.108071666721303,0.147007630697039,5.0,0
11660,19914.0,1.34816725996528,-0.531812642089039,0.737667065413582,-0.312409779034368,-1.14776721368531,-0.585711320970058,-0.810411843313548,-0.0621158

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bronze_fraude_com_id AS
SELECT
  monotonically_increasing_id() AS row_id,
  *
FROM workspace.default.bronze_fraude

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM bronze_fraude_com_id
WHERE Time = 19914 AND Amount = 5 AND Class = 0

row_id,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
11647,19914.0,-1.03448232595765,0.75994473694329,1.34467333214426,-1.55424449313282,-0.532825515302296,0.220303026209161,-1.1655522307925,-2.48709461592401,0.0548889853728043,-0.668947069379951,1.25724392043396,-2.84372861554008,0.281256306188759,1.57701133182995,-2.52618507333622,1.19765266994599,0.983546058096941,-0.640644599643313,0.41662034442768,0.689235733396798,-1.37178854451404,0.378573058124428,-0.0397428935881618,-0.0807766069833094,0.0706217761108173,-0.440477283618114,0.263666044896942,0.179713530514989,5.0,0
11649,19914.0,1.40569296761931,-0.45505761122312,0.640889499952696,-0.248345539971386,-1.06801849497726,-0.784040104907807,-0.682285157753401,-0.220919295343124,0.91299803363428,0.136481991960182,0.275076275895739,-3.08935555714058,1.21398251360653,1.05926007666205,-1.17581338725582,0.56842548632104,1.43895793688241,-1.43970475736393,0.740562563593792,-0.0373660151840986,-0.180267477527653,-0.190712759947457,-0.0108780888913218,0.351078610374535,0.52043600996309,-0.281235288578146,-0.0150176585891398,0.0050656411612116,5.0,0
11650,19914.0,1.42411717796772,-0.495474997994606,0.308426512499935,-0.572271349033493,-0.546665211090953,0.0475671039489355,-0.772449390676845,-0.0445350182606686,0.642803283224896,0.293055519790533,1.27775401363277,-2.44113221832095,1.74244923959007,1.11205980883751,-1.86356243531288,0.995799141393401,0.846966499021451,-0.718355220387906,1.35341543949661,0.0313467832666263,-0.180209004650092,-0.199961441135866,-0.135928258264616,-0.557044785711034,0.621530773785195,-0.24653179455173,-0.0169545074053346,-0.0129047743224621,5.0,0
11653,19914.0,1.27601095064793,-0.547342131264456,0.62393594525837,-0.470525045960677,-0.568852834920904,0.501482071827603,-0.889601840299811,0.161101329661102,0.529065727003975,0.193980210542405,2.55618983581269,-1.8185312891651,1.85618778466974,1.16045801483729,-1.45402305619153,0.13821088569798,1.70709390403971,-1.94430949984752,0.301017752312921,-0.041218572529309,-0.0636808146143241,0.216159253930332,0.0277339125115204,-0.300367929498332,0.368807159237695,-0.241935998662329,0.0193671411111679,-0.0102572983572635,5.0,0
11654,19914.0,1.3832213319992,-0.374322572705347,0.8013874878582,-0.179139278300606,-1.14362383379193,-0.955257153072016,-0.606964814357481,-0.29996924333716,0.740698066505206,0.107019407517051,0.736722662801087,-2.45338201895804,2.14797746716525,0.881791989037497,-1.3093229465559,0.42611786519335,1.50151663195615,-1.66862843688839,0.610509107434491,0.0150100714229332,-0.147277553262847,-0.0482778196122065,0.0169406300812118,0.691634394346714,0.510036518980358,-0.304183368130678,-0.0080145821287395,0.0103777535617432,5.0,0
11655,19914.0,1.4043288801027,-0.589821574354749,0.602211985568437,-0.230269357857081,-1.23403317374756,-0.82829818125613,-0.808491432755844,-0.115237832167687,1.19466196656086,0.191118022985743,-0.0421574993573741,-4.07868735763338,-0.646228547886017,1.42946514555525,-0.971716342700956,0.655684208087718,1.54525856663996,-1.21147921257323,0.753121944273782,-0.156550696880254,-0.201655393179536,-0.343059795413449,0.0114499383763353,0.30014764351831,0.457494760713557,-0.275001589356924,-0.0271783120673688,0.0010705063139755,5.0,0
11659,19914.0,-0.515425742222772,-0.377471797494264,1.90381634528886,-0.0719693139416316,-0.314139757363762,-1.11685454251755,-0.583850048889903,-0.092523558016707,0.285986382992853,0.0263967244759626,0.398356288771526,-3.5045123663706,0.799950592328616,1.16251737272034,-0.834158964197737,-0.0147692798163684,1.95854975262699,-1.17903988696488,1.74932000589951,0.272848068134969,0.003110215406888,0.148909378887346,0.0430113094247351,0.672869881084047,-0.328234524169491,-0.228852257060805,0.108071666721303,0.147007630697039,5.0,0
11660,19914.0,1.34816725996528,-0.531812642089039,0.737667065413582,-0.312409779034368,-1.14776721368531,-0.585711320970058,-0.810411843313548,-0.0621158

**Nota:** consultando `bronze_fraude_com_id` já como tabela persistente, os `row_id` voltam a ser os mesmos entre execuções (ex.: 11647, 11649, 11650...), confirmando que agora são estáveis e não mudam mais a cada run — diferente do que acontecia com `monotonically_increasing_id()` sobre uma view/consulta não persistida.

Com um identificador único e estável por transação, o próximo passo é remover as 1081 duplicatas reais, mantendo apenas uma cópia de cada grupo repetido.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_fraude AS
SELECT * FROM (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY Time, Amount, Class, V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,
                   V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28
      ORDER BY row_id
    ) AS rn
  FROM workspace.default.bronze_fraude_com_id
)
WHERE rn = 1

num_affected_rows,num_inserted_rows


In [0]:
%sql
Select count(*) as total from silver_fraude 

total
283726


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM workspace.default.bronze_fraude_com_id;

total_registros
284807


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.default.bronze_fraude_com_id) AS total_bronze,
  (SELECT COUNT(*) FROM workspace.default.silver_fraude) AS total_silver,
  (SELECT COUNT(*) FROM workspace.default.bronze_fraude_com_id) -
  (SELECT COUNT(*) FROM workspace.default.silver_fraude) AS linhas_removidas

total_bronze,total_silver,linhas_removidas
284807,283726,1081


In [ ]:
from pyspark.sql import functions as F

df_silver = spark.table("workspace.default.silver_fraude")

# Calcula média e desvio padrão de Amount e Time
stats = df_silver.select(
    F.mean("Amount").alias("amount_mean"), F.stddev("Amount").alias("amount_std"),
    F.mean("Time").alias("time_mean"), F.stddev("Time").alias("time_std")
).collect()[0]

# Aplica a padronização manualmente: (x - média) / desvio
df_scaled = df_silver.withColumn(
    "Amount_scaled", (F.col("Amount") - stats["amount_mean"]) / stats["amount_std"]
).withColumn(
    "Time_scaled", (F.col("Time") - stats["time_mean"]) / stats["time_std"]
)

df_scaled.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_fraude")


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.mlflow_temp

In [ ]:
import mlflow
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from mlflow.tracking import MlflowClient

# 1. Ler a Gold e montar o vetor de features
df_gold = spark.table("workspace.default.gold_fraude")
colunas_features = [f"V{i}" for i in range(1, 29)] + ["Amount_scaled", "Time_scaled"]
assembler = VectorAssembler(inputCols=colunas_features, outputCol="features")
df_gold_com_features = assembler.transform(df_gold)

# 2. Split treino/teste
df_treino, df_teste = df_gold_com_features.randomSplit([0.8, 0.2], seed=42)

# 3. Treino + registro do modelo, tudo dentro do mesmo run
with mlflow.start_run(run_name="random_forest_fraude") as run:
    rf = RandomForestClassifier(
        labelCol="Class",
        featuresCol="features",
        numTrees=100,
        maxDepth=10,
        seed=42
    )
    modelo_rf = rf.fit(df_treino)
    previsoes = modelo_rf.transform(df_teste)

    avaliador_auc = BinaryClassificationEvaluator(labelCol="Class", metricName="areaUnderPR")
    auc_pr = avaliador_auc.evaluate(previsoes)

    avaliador_f1 = MulticlassClassificationEvaluator(labelCol="Class", metricName="f1")
    f1 = avaliador_f1.evaluate(previsoes)
    mlflow.log_metric("auc_pr", auc_pr)
    mlflow.log_metric("f1", f1)

    avaliador_f1_fraude = MulticlassClassificationEvaluator(
        labelCol="Class", metricName="fMeasureByLabel",
        predictionCol="prediction", metricLabel=1.0
    )
    f1_fraude = avaliador_f1_fraude.evaluate(previsoes)
    mlflow.log_metric("f1_fraude", f1_fraude)
    print(f"O f1_fraude (classe fraude): {f1_fraude:.4f}")

    from mlflow.models import infer_signature
    signature = infer_signature(df_treino.select("features"), previsoes.select("prediction"))
    mlflow.spark.log_model(
        modelo_rf,
        "modelo_fraude",
        dfs_tmpdir="/Volumes/workspace/default/mlflow_temp",
        signature=signature
    )
    previsoes.groupBy("Class", "prediction").count().orderBy("Class", "prediction").show()
    run_id = run.info.run_id
    print(f"Run finalizada: {run_id} | AUC-PR: {auc_pr:.4f} | F1: {f1:.4f}")

# 4. Registrar o modelo (fora do bloco 'with', run já fechada)
client = MlflowClient()
model_uri = f"runs:/{run_id}/modelo_fraude"

try:
    model_details = mlflow.register_model(model_uri=model_uri, name="workspace.default.modelo_fraude_cartao")
    nome_modelo = model_details.name
    is_unity_catalog = True
except Exception as e:
    print(f"Aviso ao usar Unity Catalog: {e}\nTentando Workspace Model Registry padrão...")
    model_details = mlflow.register_model(model_uri=model_uri, name="modelo_fraude_cartao")
    nome_modelo = model_details.name
    is_unity_catalog = False

print(f"Modelo registrado: {nome_modelo} (versão {model_details.version})")


In [ ]:
# Promove o modelo para Produção somente se atingir o mínimo de qualidade (AUC-PR)
if auc_pr >= 0.80:
    if is_unity_catalog:
        client.set_registered_model_alias(name=nome_modelo, alias="Production", version=model_details.version)
        print(f"[Unity Catalog] Versão {model_details.version} promovida a 'Production'!")
    else:
        client.transition_model_version_stage(
            name=nome_modelo,
            version=model_details.version,
            stage="Production",
            archive_existing_versions=True
        )
        print(f"[Workspace Registry] Versão {model_details.version} movida para 'Production'!")
else:
    print(f"Modelo rejeitado! AUC-PR {auc_pr:.2f} é menor que o mínimo de 0.80.")


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.dim_tempo AS
SELECT DISTINCT
  Time,
  CAST(Time / 3600 AS INT) % 24 AS hora_do_dia,
  CASE 
    WHEN CAST(Time / 3600 AS INT) % 24 BETWEEN 0 AND 5 THEN 'Madrugada'
    WHEN CAST(Time / 3600 AS INT) % 24 BETWEEN 6 AND 11 THEN 'Manhã'
    WHEN CAST(Time / 3600 AS INT) % 24 BETWEEN 12 AND 17 THEN 'Tarde'
    ELSE 'Noite'
  END AS periodo_do_dia
FROM workspace.default.silver_fraude;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.fato_transacoes AS
SELECT
  Time,
  Amount,
  Class AS is_fraude
FROM workspace.default.silver_fraude;

num_affected_rows,num_inserted_rows


In [0]:
%sql
OPTIMIZE workspace.default.fato_transacoes
ZORDER BY (is_fraude);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1215595), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1787315221142, 1787315222269, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql
SELECT
  d.periodo_do_dia,
  COUNT(*) AS total_transacoes,
  SUM(f.is_fraude) AS total_fraudes,
  ROUND(SUM(f.is_fraude) / COUNT(*) * 100, 4) AS taxa_fraude_pct,
  ROUND(AVG(f.Amount), 2) AS valor_medio
FROM workspace.default.fato_transacoes f
JOIN workspace.default.dim_tempo d ON f.Time = d.Time
GROUP BY d.periodo_do_dia
ORDER BY taxa_fraude_pct DESC;

periodo_do_dia,total_transacoes,total_fraudes,taxa_fraude_pct,valor_medio
Madrugada,23842,115,0.4823,61.34
Manhã,70643,118,0.167,98.38
Tarde,96121,133,0.1384,102.16
Noite,93120,107,0.1149,73.78


In [0]:
%sql
SELECT 
  CASE WHEN is_fraude = 1 THEN 'Fraude' ELSE 'Legítima' END AS tipo,
  ROUND(AVG(Amount), 2) AS valor_medio,
  ROUND(MIN(Amount), 2) AS valor_min,
  ROUND(MAX(Amount), 2) AS valor_max
FROM workspace.default.fato_transacoes
GROUP BY is_fraude;

tipo,valor_medio,valor_min,valor_max
Legítima,88.41,0.0,25691.16
Fraude,123.87,0.0,2125.87


In [0]:
%sql
SELECT
  COUNT(*) AS total_transacoes,
  SUM(is_fraude) AS total_fraudes,
  ROUND(SUM(is_fraude) / COUNT(*) * 100, 4) AS taxa_geral_pct
FROM workspace.default.fato_transacoes;

total_transacoes,total_fraudes,taxa_geral_pct
283726,473,0.1667
